# Transformasi Sumbu Y menggunakan Translasi dan Pencerminan

**Blok Utama** (Biru, Ungu, Oranye, Hijau): posisi asli titik-titik koordinat GeoGebra Anda.  
**Blok Cermin** (Merah): hasil setelah mengalami translasi (pergeseran vertikal ke atas) dan refleksi (pencerminan atas-bawah terhadap sumbu X).

### Persamaan Matematika (Koordinat Homogen):
1. **Translasi ke atas sejauh $t$ satuan ($t_x = 0, t_y = t$):**
   $$T = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & t \\ 0 & 0 & 1 \end{bmatrix}$$
2. **Refleksi terhadap Sumbu X / Atas-Bawah ($s_y = s$, dengan $s$ turun dari $1 \to -1$):**
   $$M = \begin{bmatrix} 1 & 0 & 0 \\ 0 & s & 0 \\ 0 & 0 & 1 \end{bmatrix}$$
3. **Transformasi Gabungan ($A = M \cdot T$):**
   $$A = \begin{bmatrix} 1 & 0 & 0 \\ 0 & s & 0 \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & t \\ 0 & 0 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & s & s \cdot t \\ 0 & 0 & 1 \end{bmatrix}$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from IPython.display import HTML, display

In [ ]:
# ── Titik-titik dari GeoGebra Anda ──────────────────────────────
%matplotlib inline
TITIK = {
    'A': (2, 3),  'B': (2, 4),  'C': (3, 4),  'D': (3, 3),
    'E': (2,-3),  'F': (3,-3),  'G': (2,-4),  'H': (3,-4),
    'I': (2, 2),  'J': (3, 2),  'K': (2, 1),  'L': (3, 1),
    'M': (2,-1),  'N': (3,-1),  'O': (2,-2),  'P': (3,-2),
}

FRAMES  = 80
PAUSE_F = 15

# ── Setup figure ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7), dpi=70)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=1.2, zorder=2)
ax.axvline(0, color='black', linewidth=1.2, zorder=2)
ax.grid(True, color='#cccccc', linewidth=0.5)
ax.tick_params(labelsize=9)
ax.set_title('Animasi Translasi ke Atas dan Pencerminan terhadap Sumbu X (Atas-Bawah)\n'
             '(Klik grafik untuk Pause/Play)', fontsize=11, pad=10)

# ── Warna per pasangan blok Anda ─────────────────────────────────
BLOK = [
    {'asli': ['A','B','C','D'], 'warna': '#1a6fb5'},
    {'asli': ['I','J','L','K'], 'warna': '#7c3aed'},
    {'asli': ['M','N','P','O'], 'warna': '#d97706'},
    {'asli': ['E','F','H','G'], 'warna': '#10b981'},
]

# artists per blok
blok_artists = []
for blok in BLOK:
    warna = blok['warna']
    # Outline asli (closed loop)
    l_asli, = ax.plot([], [], color=warna, lw=2, zorder=5)
    # Dot asli
    d_asli, = ax.plot([], [], 'o', color=warna, ms=7, zorder=6)
    # Teks asli
    t_asli = [ax.text(0, 0, lbl, color=warna, fontsize=8,
                       fontweight='bold', zorder=7)
               for lbl in blok['asli']]

    # Outline cermin (closed loop)
    l_cermin, = ax.plot([], [], color='#e74c3c', lw=2, zorder=5)
    # Dot cermin
    d_cermin, = ax.plot([], [], 'o', color='#e74c3c', ms=7, zorder=6)
    # Teks cermin
    t_cermin = [ax.text(0, 0, lbl+"'", color='#e74c3c', fontsize=8,
                          fontweight='bold', zorder=7)
                  for lbl in blok['asli']]

    blok_artists.append({
        'l_asli': l_asli, 'd_asli': d_asli, 't_asli': t_asli,
        'l_cermin': l_cermin, 'd_cermin': d_cermin, 't_cermin': t_cermin,
        'asli_names': blok['asli']
    })

# ── Info teks matriks ────────────────────────────────────────────
mat_txt = ax.text(-5.8, 4.0, '', fontsize=9, color='#222',
                  fontfamily='monospace',
                  bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f4ff',
                            edgecolor='#aaaaaa', alpha=0.9))

# ── Legend ───────────────────────────────────────────────────────
legend_elements = [
    Line2D([0],[0], color='#1a6fb5', lw=2, label='Blok Asli'),
    Line2D([0],[0], color='#e74c3c', lw=2, label="Blok Cermin (P')"),
]
ax.legend(handles=legend_elements, loc='upper right',
          fontsize=9, framealpha=0.9)

# ── State pause ──────────────────────────────────────────────────
state = {'paused': False}
def on_click(event):
    if event.inaxes == ax:
        state['paused'] = not state['paused']
fig.canvas.mpl_connect('button_press_event', on_click)

# ── Easing scale dan translasi ───────────────────────────────────
def get_scale_and_trans(frame):
    if frame < PAUSE_F:
        return 0.0, 1.0
    if frame >= FRAMES - PAUSE_F:
        return 1.0, -1.0

    mid = FRAMES // 2
    if frame < mid:
        t_val = (frame - PAUSE_F) / (mid - PAUSE_F - 1)
        t_val = 3*t_val**2 - 2*t_val**3
        return t_val, 1.0
    else:
        s_val = (frame - mid) / (FRAMES - PAUSE_F - mid)
        s_val = 3*s_val**2 - 2*s_val**3
        s = 1.0 - 2.0 * s_val
        return 1.0, s

cur_frame = [0]

def animate(frame):
    if state['paused']:
        frame = cur_frame[0]
    else:
        cur_frame[0] = frame

    t, s = get_scale_and_trans(frame)
    all_artists = [mat_txt]

    for ba in blok_artists:
        # Get original coordinates for this block
        pts = [TITIK[k] for k in ba['asli_names']]
        xs_a = [p[0] for p in pts]
        ys_a = [p[1] for p in pts]

        # Close loop for outline asli
        xs_a_loop = xs_a + [xs_a[0]]
        ys_a_loop = ys_a + [ys_a[0]]
        ba['l_asli'].set_data(xs_a_loop, ys_a_loop)
        ba['d_asli'].set_data(xs_a, ys_a)

        # Dynamic offsets for original labels
        xmin_a, xmax_a = min(xs_a), max(xs_a)
        ymin_a, ymax_a = min(ys_a), max(ys_a)
        for tx, x, y in zip(ba['t_asli'], xs_a, ys_a):
            dx = -0.18 if x == xmin_a else 0.06
            dy = -0.22 if y == ymin_a else 0.06
            tx.set_position((x + dx, y + dy))

        # Transformed coordinates
        xs_c = xs_a
        ys_c = [s * (y + t) for y in ys_a]

        # Close loop for outline cermin
        xs_c_loop = xs_c + [xs_c[0]]
        ys_c_loop = ys_c + [ys_c[0]]
        ba['l_cermin'].set_data(xs_c_loop, ys_c_loop)
        ba['d_cermin'].set_data(xs_c, ys_c)

        # Dynamic offsets for cermin labels (handles flipping dynamically!)
        xmin_c, xmax_c = min(xs_c), max(xs_c)
        ymin_c, ymax_c = min(ys_c), max(ys_c)
        for tx, x, y in zip(ba['t_cermin'], xs_c, ys_c):
            dx = -0.18 if x == xmin_c else 0.06
            dy = -0.22 if y == ymin_c else 0.06
            tx.set_position((x + dx, y + dy))

        all_artists += [ba['l_asli'], ba['d_asli']] + ba['t_asli']
        all_artists += [ba['l_cermin'], ba['d_cermin']] + ba['t_cermin']

    mat_txt.set_text(
        f'T_trans = [1.0  0.0  0.00]\n'
        f'          [0.0  1.0  {t:.2f}]\n'
        f'          [0.0  0.0  1.00]\n\n'
        f'M_refly = [1.0  0.0  0.00]\n'
        f'          [0.0  {s:.2f}  0.00]\n'
        f'          [0.0  0.0  1.00]\n\n'
        f'A_gab   = [1.0  0.0  0.00]\n'
        f'          [0.0  {s:.2f}  {s*t:.2f}]\n'
        f'          [0.0  0.0  1.00]'
    )
    return all_artists

ani = animation.FuncAnimation(
    fig, animate, frames=FRAMES,
    interval=45, blit=False, repeat=True
)

import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 100  # supaya tidak terpotong

plt.tight_layout()
js_html = ani.to_jshtml()
plt.close(fig)
display(HTML(js_html))


## Tabel Koordinat Transformasi Sumbu Y

| Titik Asli | Hasil Translasi $T(y+1)$ | Hasil Akhir Refleksi $P''(-y-1)$ |
|:---:|:---:|:---:|
| $A(2,3)$ | $A'(2,4)$ | $A''(2,-4)$ |
| $B(2,4)$ | $B'(2,5)$ | $B''(2,-5)$ |
| $C(3,4)$ | $C'(3,5)$ | $C''(3,-5)$ |
| $D(3,3)$ | $D'(3,4)$ | $D''(3,-4)$ |
| $I(2,2)$ | $I'(2,3)$ | $I''(2,-3)$ |
| $J(3,2)$ | $J'(3,3)$ | $J''(3,-3)$ |
| $K(2,1)$ | $K'(2,2)$ | $K''(2,-2)$ |
| $L(3,1)$ | $L'(3,2)$ | $L''(3,-2)$ |
| $M(2,-1)$ | $M'(2,0)$ | $M''(2,0)$ |
| $N(3,-1)$ | $N'(3,0)$ | $N''(3,0)$ |
| $O(2,-2)$ | $O'(2,-1)$ | $O''(2,1)$ |
| $P(3,-2)$ | $P'(3,-1)$ | $P''(3,1)$ |
| $E(2,-3)$ | $E'(2,-2)$ | $E''(2,2)$ |
| $F(3,-3)$ | $F'(3,-2)$ | $F''(3,2)$ |
| $G(2,-4)$ | $G'(2,-3)$ | $G''(2,3)$ |
| $H(3,-4)$ | $H'(3,-3)$ | $H''(3,3)$ |
